# E1 tail — cache C1_s43 train+val features

One short GPU session: extract d_enc features from `C1_s43/best.ckpt`
over the frozen p2_lab **train and val** sets via
`harness.cache_features` (the exact same path every probe session used
— no new logic). Writes `features_best_train.npz` /
`features_best_val.npz` next to the checkpoint on Drive.

**Why this cache exists** (STATUS 2026-07-18, both uses pre-declared):
1. the **C1⊕C1′ ensemble control** of the §7 concat diagnostic — the
   number that separates "SupCon knows something CE doesn't" from
   "any second encoder helps";
2. the **NCM/kNN seed-robustness footnote** (C1 only, declared
   asymmetry — the entry already sits in the NCM/kNN notebook and
   un-SKIPs itself once this cache lands).

Train/val only — **no test contact** (§0.7). Expected sizes: train
53400 samples, val 1396, d=256 (the asserts below check this).
Cost: ~2 min of forward passes on a T4 after staging.

## Environment setup

Full preamble incl. data staging (unlike the diagnostics notebooks:
`cache_features` runs the encoder over the real staged data). GPU
runtime recommended.

In [ ]:
from pathlib import Path
import subprocess
import sys
import torch
import zipfile

REPO_URL = "https://github.com/FilippoIsoni/sharp-har.git"

cwd = Path.cwd().resolve()
if (cwd / "sharp_har").exists():
    REPO_DIR = cwd
elif (cwd.parent / "sharp_har").exists():
    REPO_DIR = cwd.parent
elif (cwd.parent.parent / "sharp_har").exists():
    REPO_DIR = cwd.parent.parent
else:
    REPO_DIR = Path("/content/sharp-har")
    if not REPO_DIR.exists():
        REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

# After the clone, so the file actually exists on a fresh runtime.
!pip install -q -r {REPO_DIR}/requirements.txt

sys.path.insert(0, str(REPO_DIR))

from google.colab import drive
from sharp_har.utils import read_yaml

paths_cfg = read_yaml(REPO_DIR / "configs" / "paths.yaml")
drive_root = Path(paths_cfg["drive_root"])
stage_dir = Path(paths_cfg["stage_dir"])
CKPT_ROOT = Path(paths_cfg["ckpt_root"])

drive.mount("/content/drive")  # idempotent

# Stage the zip archives if needed (same convention as 00/03/04).
if not (stage_dir.exists() and any(stage_dir.rglob("*.txt"))):
    stage_dir.mkdir(parents=True, exist_ok=True)
    for zip_name in paths_cfg["zips"]:
        src = drive_root / zip_name
        dst = Path("/content") / zip_name
        print(f"copying {src} -> {dst}")
        subprocess.run(["cp", str(src), str(dst)], check=True)
        with zipfile.ZipFile(dst) as zf:
            zf.extractall(stage_dir)

print("Repo dir:", REPO_DIR)
print("GPU available:", torch.cuda.is_available())
print("Stage dir:", stage_dir)
print("Checkpoint root:", CKPT_ROOT)

## Cache train + val features from C1_s43/best.ckpt

`cache_features` skips nothing silently: if a cache file already exists
at the target path it is simply overwritten by a fresh, identical
extraction (frozen encoder, no augmentation — §5.3 note i), so
re-running this cell is safe.

In [ ]:
import numpy as np
from sharp_har.harness import cache_features

CKPT = CKPT_ROOT / "C1_s43" / "best.ckpt"
P2_SPLIT = REPO_DIR / "splits" / "p2_lab.json"
assert CKPT.exists(), f"{CKPT} not found - is the C1_s43 Drive folder yours or shortcut-linked?"

EXPECTED = {"train": 53400, "val": 1396}  # frozen p2_lab sample counts
paths = {}
for set_name in ("train", "val"):
    paths[set_name] = cache_features(
        CKPT, P2_SPLIT, set_name, stage_dir=stage_dir, repo_dir=REPO_DIR,
    )
    d = np.load(paths[set_name], allow_pickle=False)
    n, dim = d["features"].shape
    assert n == EXPECTED[set_name], f"{set_name}: {n} samples, expected {EXPECTED[set_name]}"
    assert dim == 256, f"{set_name}: d={dim}, expected 256"
    print(f"{set_name}: {n} samples, d={dim} -> {paths[set_name]}")

print("\nDone - both caches on Drive. This unblocks:")
print("1. the concat C1(+)C1' ensemble control (concat diagnostic session);")
print("2. the C1_s43 NCM/kNN footnote entry (rerun that notebook: the SKIP resolves itself).")

## Archiving

Download the executed copy and commit it verbatim to `notebooks/runs/`
as `YYYY-MM-DD_c1_s43_feature_cache.ipynb` + STATUS line, same commit.
This template stays output-free.